In [7]:
%pip install azure-search-documents azure-identity dotenv 

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


# 1. Importaciones

In [1]:
import os
from dotenv import load_dotenv
from azure.core.credentials import AzureKeyCredential
from azure.search.documents.indexes import SearchIndexClient
from azure.search.documents.indexes.models import (
    SearchIndex,
    SearchField,
    SearchFieldDataType,
    SimpleField,
    SearchableField,
    VectorSearch,
    HnswAlgorithmConfiguration,
    VectorSearchProfile,
    SemanticConfiguration,
    SemanticPrioritizedFields,
    SemanticField,
    SemanticSearch
)

# 2. Configuración

In [3]:
load_dotenv()
service_endpoint = os.getenv("AZURE_SEARCH_ENDPOINT")
admin_key = os.getenv("AZURE_SEARCH_KEY")
index_name = "ponal-documents-index"
vector_dimensions = 3072

# 3. Definición del índice

In [ ]:
fields = [
    # Document identifiers
    SimpleField(
        name="id",
        type=SearchFieldDataType.String,
        key=True,
        filterable=True,
        sortable=True
    ),
    SearchableField(
        name="chunk_id",
        type=SearchFieldDataType.String,
        filterable=True,
        sortable=True
    ),
    
    # Document content (main searchable field)
    SearchableField(
        name="content",
        type=SearchFieldDataType.String,
        searchable=True,
        analyzer_name="es.microsoft"  # Spanish analyzer
    ),
    
    # Document metadata
    SearchableField(
        name="filename",
        type=SearchFieldDataType.String,
        searchable=True,
        filterable=True,
        facetable=True,
        sortable=True
    ),
    SimpleField(
        name="filepath",
        type=SearchFieldDataType.String,
        filterable=True,
        sortable=True
    ),
    SimpleField(
        name="file_type",
        type=SearchFieldDataType.String,
        filterable=True,
        facetable=True,
        sortable=True
    ),
    SimpleField(
        name="blob_url",
        type=SearchFieldDataType.String,
        filterable=True
    ),
    
    # Structure Reference
    SearchableField(
        name="section",
        type=SearchFieldDataType.String,
        searchable=True,
        filterable=True,
        facetable=True
    ),
    
    # Document structure info
    SimpleField(
        name="pages",
        type=SearchFieldDataType.Int32,
        filterable=True,
        sortable=True
    ),
    SimpleField(
        name="pages_total",
        type=SearchFieldDataType.Int32,
        filterable=True
    ),
    SimpleField(
        name="page_start",
        type=SearchFieldDataType.Int32,
        filterable=True,
        sortable=True
    ),
    SimpleField(
        name="page_end",
        type=SearchFieldDataType.Int32,
        filterable=True
    ),
    SimpleField(
        name="chunk_index",
        type=SearchFieldDataType.Int32,
        filterable=True,
        sortable=True
    ),
    SimpleField(
        name="total_chunks",
        type=SearchFieldDataType.Int32,
        filterable=True
    ),
    
    # Vector embeddings for semantic search
    SearchField(
        name="embedding",
        type=SearchFieldDataType.Collection(SearchFieldDataType.Single),
        searchable=True,
        vector_search_dimensions=vector_dimensions,
        vector_search_profile_name="myHnswProfile"
    ),
]

# Configure HNSW vector search algorithm
vector_search = VectorSearch(
    algorithms=[
        HnswAlgorithmConfiguration(
            name="myHnsw",
            parameters={
                "m": 4,
                "efConstruction": 400,
                "efSearch": 500,
                "metric": "cosine"
            }
        )
    ],
    profiles=[
        VectorSearchProfile(
            name="myHnswProfile",
            algorithm_configuration_name="myHnsw"
        )
    ]
)

# Configure semantic search for better relevance
semantic_config = SemanticConfiguration(
    name="my-semantic-config",
    prioritized_fields=SemanticPrioritizedFields(
        content_fields=[SemanticField(field_name="content")],
        keywords_fields=[SemanticField(field_name="filename")]
    )
)

semantic_search = SemanticSearch(configurations=[semantic_config])

# Create the index definition
index = SearchIndex(
    name=index_name,
    fields=fields,
    vector_search=vector_search,
    semantic_search=semantic_search
)

print("Índice definido correctamente")
print(f"Nombre del índice: {index_name}")
print(f"Número de campos: {len(fields)}")

Índice definido correctamente
Nombre del índice: ponal-documents-index
Número de campos: 15


# 4. Creación del índice

In [6]:
try:
    # Crear cliente para gestión de índices
    credential = AzureKeyCredential(admin_key)
    index_client = SearchIndexClient(endpoint=service_endpoint, credential=credential)
    
    # Crear el índice
    print(f"Creando índice '{index_name}' en Azure Cognitive Search...")
    result = index_client.create_index(index)
    print(f"✅ Índice '{index_name}' creado exitosamente!")
    
    # Mostrar información del índice creado
    print(f"\nInformación del índice creado:")
    print(f"- Nombre: {result.name}")
    print(f"- Campos: {len(result.fields)}")
    print(f"- Búsqueda vectorial: {'Habilitada' if result.vector_search else 'Deshabilitada'}")
    print(f"- Búsqueda semántica: {'Habilitada' if result.semantic_search else 'Deshabilitada'}")
    
except Exception as e:
    print(f"❌ Error al crear el índice: {str(e)}")
    print("Verifica las credenciales y conexión con Azure Cognitive Search")

Creando índice 'ponal-documents-index' en Azure Cognitive Search...
✅ Índice 'ponal-documents-index' creado exitosamente!

Información del índice creado:
- Nombre: ponal-documents-index
- Campos: 15
- Búsqueda vectorial: Habilitada
- Búsqueda semántica: Habilitada


# 5. Verificación del índice creado

In [6]:
try:
    # Crear cliente para gestión de índices
    credential = AzureKeyCredential(admin_key)
    index_client = SearchIndexClient(endpoint=service_endpoint, credential=credential)
    
    # Listar todos los índices disponibles
    print("Verificando índices disponibles en el servicio...")
    indexes = list(index_client.list_index_names())
    
    print(f"\nTotal de índices encontrados: {len(indexes)}")
    
    if index_name in indexes:
        print(f"✅ Índice '{index_name}' verificado y presente en el servicio")
        
        # Obtener detalles del índice específico
        print(f"\nObteniendo detalles del índice '{index_name}'...")
        created_index = index_client.get_index(index_name)
        
        print(f"✅ Detalles del índice '{index_name}':")
        print(f"  - Nombre: {created_index.name}")
        print(f"  - Campos: {len(created_index.fields)}")
        
        # Listar nombres de campos
        print("  - Lista de campos:")
        for field in created_index.fields:
            print(f"    • {field.name} ({field.type})")
        
        # Verificar configuración de búsqueda vectorial
        if created_index.vector_search:
            print(f"  - Búsqueda vectorial: Habilitada")
            print(f"    • Perfiles: {len(created_index.vector_search.profiles)}")
            print(f"    • Algoritmos: {len(created_index.vector_search.algorithms)}")
        
        # Verificar configuración de búsqueda semántica
        if created_index.semantic_search:
            print(f"  - Búsqueda semántica: Habilitada")
            print(f"    • Configuraciones: {len(created_index.semantic_search.configurations)}")
        
    else:
        print(f"❌ Índice '{index_name}' NO encontrado en el servicio")
        print(f"Índices disponibles: {indexes}")
        
except Exception as e:
    print(f"❌ Error al verificar el índice: {str(e)}")

Verificando índices disponibles en el servicio...

Total de índices encontrados: 1
✅ Índice 'ponal-documents-index' verificado y presente en el servicio

Obteniendo detalles del índice 'ponal-documents-index'...
✅ Detalles del índice 'ponal-documents-index':
  - Nombre: ponal-documents-index
  - Campos: 15
  - Lista de campos:
    • id (Edm.String)
    • chunk_id (Edm.String)
    • content (Edm.String)
    • filename (Edm.String)
    • filepath (Edm.String)
    • file_type (Edm.String)
    • blob_url (Edm.String)
    • section (Edm.String)
    • pages (Edm.Int32)
    • pages_total (Edm.Int32)
    • page_start (Edm.Int32)
    • page_end (Edm.Int32)
    • chunk_index (Edm.Int32)
    • total_chunks (Edm.Int32)
    • embedding (Collection(Edm.Single))
  - Búsqueda vectorial: Habilitada
    • Perfiles: 1
    • Algoritmos: 1
  - Búsqueda semántica: Habilitada
    • Configuraciones: 1
